In [ ]:
#from client_setup import model, client
from helper_chat import chat, add_user_message, add_assistant_message
from helper_prompt_evaluator import PromptEvaluator

DATASET_FILENAME = 'dataset_prompting.json'

In [ ]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [ ]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    # Where to write the generated dataset
    output_file=DATASET_FILENAME,
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
    # force_regenerate=True,  # uncomment to overwrite existing dataset
)

In [ ]:
# define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case

def run_prompt(prompt_input):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.
    <athlete_information>
        - Height: {prompt_input["height"]}
        - Weight: {prompt_input["weight"]}
        - Goal: {prompt_input["goal"]}
        - Dietary restrictions: {prompt_input["restrictions"]}
    </athlete_information>
Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts  
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned

Here is an sample of example input and an ideal output:
<sample_input>
height: 180 cm
weight: 72 kg
goal: Marathon training with carbohydrate loading and maximum caloric intake for endurance performance
restrictions: None
</sample_input>
<ideal_output>
# One-Day Marathon Training Meal Plan
**Athlete: 180 cm, 72 kg | Marathon Training Phase**

---

## Daily Nutritional Targets
- **Calories:** 3,200-3,500 kcal
- **Protein:** 120-130g (15%)
- **Carbohydrates:** 520-560g (65%)
- **Fat:** 85-95g (25%)

---

## MEAL PLAN

### **BREAKFAST** (7:00 AM)
*Pre-training fuel*
- Oatmeal: 80g dry
- Banana: 150g
- Honey: 30g
- Whole milk: 250ml
- Almonds: 25g

**Macros:** 650 cal | 18g protein | 110g carbs | 15g fat

---

### **MID-MORNING SNACK** (10:00 AM)
*Post-training recovery*
- Greek yogurt: 200g
- Granola: 50g
- Blueberries: 100g
- Honey drizzle: 15g

**Macros:** 380 cal | 20g protein | 55g carbs | 8g fat

---

### **LUNCH** (1:00 PM)
*Carb-loading focus*
- Brown rice: 200g cooked
- Grilled chicken breast: 150g
- Olive oil: 15ml
- Mixed vegetables (broccoli, carrots): 150g
- Whole wheat bread: 60g

**Macros:** 750 cal | 42g protein | 95g carbs | 18g fat

---

### **AFTERNOON SNACK** (4:00 PM)
*Energy boost*
- White bread: 80g
- Peanut butter: 30g
- Apple: 180g
- Sports drink (6% carbs): 500ml

**Macros:** 580 cal | 16g protein | 85g carbs | 18g fat

---

### **DINNER** (7:00 PM)
*Evening recovery*
- Pasta (whole wheat): 180g cooked
- Lean ground beef: 120g
- Tomato sauce: 150g
- Parmesan cheese: 20g
- Olive oil: 10ml
- Side salad with dressing: 100g

**Macros:** 750 cal | 35g protein | 95g carbs | 20g fat

---

### **EVENING SNACK** (9:30 PM)
*Before bed*
- Cottage cheese: 150g
- Granola: 30g
- Honey: 15g

**Macros:** 280 cal | 20g protein | 35g carbs | 6g fat

---

## **DAILY TOTALS**
- **Calories:** 3,390 kcal ✓
- **Protein:** 151g (18%)
- **Carbohydrates:** 475g (56%)
- **Fat:** 85g (23%)

---

## **HYDRATION NOTES**
- Water: 3-4 liters throughout day
- Sports drink during long training sessions (60-90 min+)
- Electrolyte drink post-workout

## **TIMING TIPS**
- Eat carbs + protein within 30-60 min post-workout
- Consume majority of carbs around training times
- Light dinner 3 hours before bed for quality sleep
</ideal_output>
The solution successfully meets all three mandatory requirements with clear presentation of daily calories, macronutrient breakdown, and detailed meals with portions and timing. The competition-day context is handled exceptionally well with appropriate meal spacing and hydration strategy. However, the macronutrient ratios deviate from the specified secondary criteria targets (carbs should be 50-55% but are 48%; protein should be 20-25% but is 24%; fats should be 20-25% but are 24%). While these deviations are minor, they represent a measurable gap from stated performance targets. The meal count is appropriate for compactness but doesn't fully utilize the 5-6 meal allowance. These are secondary criterion issues that prevent a perfect score but do not trigger mandatory requirement failure.
    """
    messages = []
    add_user_message(messages,prompt)
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation (
    run_prompt_function=run_prompt, 
    dataset_file=DATASET_FILENAME, 
    extra_criteria=""" 
    The output should include 
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions and timing
    """,
    json_output_file="output_prompting.json",
    html_output_file="output_prompting.html"
)